In [1]:
import roboticstoolbox as rtb
import numpy as np
from spatialmath import SE3

In [2]:
# Define Links
link1 = rtb.RevoluteDH(d=0.15185, a=0.0, alpha=np.pi/2)
link2 = rtb.RevoluteDH(d=0.0, a=-0.24355, alpha=0.0)
link3 = rtb.RevoluteDH(d=0.0, a=-0.2132, alpha=0.0)
link4 = rtb.RevoluteDH(d=0.13105, a=-0.0, alpha=np.pi/2)
link5 = rtb.RevoluteDH(d=0.08535, a=-0.0, alpha=-np.pi/2)
link6 = rtb.RevoluteDH(d=0.0921, a=-0.0, alpha=0.0)

# Create the Robot Object
ur3e = rtb.DHRobot([link1, link2, link3, link4, link5, link6], name="UR3e")

print(ur3e)

DHRobot: UR3e, 6 joints (RRRRRR), dynamics, standard DH parameters
┌─────┬─────────┬─────────┬────────┐
│ θⱼ  │   dⱼ    │   aⱼ    │   ⍺ⱼ   │
├─────┼─────────┼─────────┼────────┤
│  q1 │  0.1519 │       0 │  90.0° │
│  q2 │       0 │ -0.2435 │   0.0° │
│  q3 │       0 │ -0.2132 │   0.0° │
│  q4 │   0.131 │      -0 │  90.0° │
│  q5 │ 0.08535 │      -0 │ -90.0° │
│  q6 │  0.0921 │      -0 │   0.0° │
└─────┴─────────┴─────────┴────────┘

┌──┬──┐
└──┴──┘



## Understanding the FK Output
The output you see above is a 4x4 Homogeneous Transformation Matrix.

   [ r11   r12   r13   tx ]
   [ r21   r22   r23   ty ]
   [ r31   r32   r33   tz ]
   [  0     0     0     1 ]

- Top-Left 3x3 (r values): Represents the Rotation (orientation) of the tool frame relative to the base.
- Right Column (tx, ty, tz): Represents the Translation (position X, Y, Z) of the tool frame relative to the base.

While matrices are great for math, they are hard for humans to read. We often convert this to XYZ (Position) and RPY (Roll-Pitch-Yaw / Orientation) for easier inspection.

In [ ]:
tcp_pos = np.array([0.2, 0.2, 0.2])
tcp_rot = np.array([0, 0, 0])

T = SE3(tcp_pos) * SE3.RPY(tcp_rot)

sol = ur3e.ikine_LM(T)
print(sol.q)

[1.26714941 2.3348515  1.4737988  0.9037385  1.57079603 0.30364693]


In [ ]:
T_check = ur3e.fkine(sol.q)
print("\nCheck Pose:\n", T_check.t)


Check Pose:
 [0.2 0.2 0.2]


In [ ]:
from roboticstoolbox import ET

class DHparam():
    def __init__(self, d, a, alpha):
        self.d = d
        self.a = a
        self.alpha = alpha

    def get_params(self):
        return self.d, self.a, self.alpha



def create_ur3e(stuck_joints: np.ndarray, stuck_angles: np.ndarray):
    DHparams = [
        DHparam(0.15185, 0, np.pi/2),
        DHparam(0.0, -0.24355, 0.0),
        DHparam(0.0, -0.2132, 0.0),
        DHparam(0.13105, 0.0, np.pi/2),
        DHparam(0.08535, 0, -np.pi/2),
        DHparam(0.0921, 0, 0)
    ]

    # Define the sequence of elementary transforms
    ets = rtb.ETS()
    for i, link in enumerate(DHparams):
        # Unpack parameters
        d, a, alpha = link.get_params()

        # Set the angle of the stuck joint if it is stuck
        if stuck_joints[i]:
            ets *= ET.Rz(stuck_angles[i])
        else:
            ets *= ET.Rz() 
        ets *= ET.tz(d) if d != 0 else ET.tx(a)
        ets *= ET.Rx(alpha)

    # Compile into an ERobot model
    ur3e = rtb.ERobot(ets, name="UR3e")

    return ur3e

def inv_kinematics(ur3e, tcp_pos: np.ndarray, tcp_rot: np.ndarray, 
                   stuck_joints: np.ndarray, stuck_angles: np.ndarray) -> np.ndarray:
    
    T = SE3(tcp_pos) * SE3.RPY(tcp_rot)
    
    # sol.q will only contain angles for the ACTIVE joints
    sol = ur3e.ikine_LM(T)
    
    # Reconstruct the full 6-DOF joint array
    full_q = np.zeros(len(stuck_joints))
    active_q_idx = 0
    
    for i, is_stuck in enumerate(stuck_joints):
        if is_stuck:
            # Inject the constant stuck angle
            full_q[i] = stuck_angles[i]
        else:
            # Map the next solved active joint angle
            full_q[i] = sol.q[active_q_idx]
            active_q_idx += 1

    return full_q

In [ ]:
stuck_joints = np.array([False, False, False, False, False, False])
stuck_angles = np.array([np.pi/4] * 6)

ur3e = create_ur3e(stuck_joints, stuck_angles)

tcp_pos = np.array([0.2, 0.2, 0.2])
tcp_rot = np.array([0.0, 0.0, 0.0])

q = inv_kinematics(ur3e, tcp_pos, tcp_rot, stuck_joints, stuck_angles)

T_check = ur3e.fkine(q)
print("\nCheck Pose:\n", T_check.t)
print("Q: ", q)


Check Pose:
 [0.2 0.2 0.2]
Q:  [1.26714922 2.33485153 1.47379869 0.90373849 1.57079654 0.30364708]
